# DuckDB Masterclass — Capítulo 04
## Leitura direta de Parquet e Friendly SQL

**Edição-piloto para autoaprendizagem e publicação** · [Série no GitHub](https://github.com/aestevaomoraes/duckdb-masterclass)

**Pré-requisitos:** SQL intermediário, Python, DuckDB e Pandas. Dados sintéticos, sem serviços externos.

**Roteiro:** entender o formato → gerar arquivo Parquet → consultar diretamente → aplicar Friendly SQL → validar as interpretações.

## 04.1 — Objetivos e diagnóstico

Ao final você poderá: distinguir leitura direta de importação persistente; consultar Parquet com `read_parquet`; utilizar `GROUP BY ALL`, `SELECT * EXCLUDE` e `SELECT * REPLACE`; e explicar o limite da expressão **zero-copy**.

**Pergunta inicial:** é necessário executar `CREATE TABLE` para analisar o conteúdo de um Parquet no DuckDB?

## 04.2 — Conceito essencial

**Ler sem importar:** o DuckDB pode consultar um arquivo Parquet diretamente, sem materializá-lo antes em uma tabela. Isso não significa que cada etapa seja *zero-copy*: leitura em disco, descompressão, buffers e conversões podem ocorrer.

```text
Arquivo Parquet (.parquet)
        | read_parquet(...)
        v
DuckDB (filtro / projeção / agregação)
        |
        v
Resultado tabular
```

`GROUP BY ALL` agrupa pelas expressões não agregadas da seleção. `EXCLUDE` remove colunas de uma projeção; `REPLACE` recalcula colunas mantendo seu nome.

## 04.3 — Preparação reproduzível

Criaremos arquivos exclusivamente em uma pasta temporária. Não há credenciais, downloads ou dados privados.

**Importante:** execute o notebook em ordem, num kernel limpo.

In [ ]:
import duckdb
import tempfile
from pathlib import Path
from IPython.display import display

con = duckdb.connect(":memory:")
pasta_temp = tempfile.TemporaryDirectory()
caminho_parquet = Path(pasta_temp.name) / "eventos.parquet"
print("DuckDB:", duckdb.__version__)
print("Arquivo de laboratório: eventos.parquet")


## 04.4 — Laboratório A: gerar um Parquet sintético

**Cenário:** o time operacional recebeu 60 eventos técnicos distribuídos em três setores.

**Hipótese:** podemos gerar um Parquet a partir de um conjunto sintético e verificar sua contagem.

In [ ]:
con.execute("""
CREATE TABLE eventos_origem AS
SELECT i AS id,
       'SETOR_' || CAST(i % 3 AS VARCHAR) AS setor,
       CAST(i % 80 + 20 AS INTEGER) AS valor
FROM range(60) t(i)
""")
con.execute(f"COPY eventos_origem TO '{caminho_parquet.as_posix()}' (FORMAT PARQUET)")
assert caminho_parquet.exists()
assert con.execute('SELECT COUNT(*) FROM eventos_origem').fetchone()[0] == 60
print('Arquivo Parquet criado; 60 eventos sintéticos.')


**Interpretação:** `COPY` gerou o arquivo uma vez; a próxima consulta será feita **diretamente no Parquet**, sem executar `CREATE TABLE` para o conteúdo lido.

## 04.5 — Laboratório B: consultar sem importar

**Hipótese:** o total de registros lidos do arquivo será igual ao total da tabela de origem.

In [ ]:
consulta = f"SELECT setor, COUNT(*) AS n, SUM(valor) AS total FROM read_parquet('{caminho_parquet.as_posix()}') GROUP BY ALL ORDER BY setor"
res_parquet = con.execute(consulta).df()
display(res_parquet)
assert res_parquet['n'].sum() == 60
print('Leitura direta validada: 60 eventos.')


**Ponto de engenharia:** leitura direta evita uma etapa de importação persistente. Não é prova de cópia zero em toda a trajetória dos bytes nem de desempenho superior sem benchmark controlado.

## 04.6 — Laboratório C: `EXCLUDE` e `REPLACE`

**Hipótese:** poderemos ocultar `id` e transformar `valor` na projeção sem alterar os dados de origem.

In [ ]:
q_exclude = f"SELECT * EXCLUDE (id) FROM read_parquet('{caminho_parquet.as_posix()}') ORDER BY setor, valor LIMIT 5"
q_replace = f"SELECT * REPLACE (valor * 2 AS valor) FROM read_parquet('{caminho_parquet.as_posix()}') ORDER BY id LIMIT 5"

res_exclude = con.execute(q_exclude).df()
res_replace = con.execute(q_replace).df()
display(res_exclude)
display(res_replace)
assert 'id' not in res_exclude.columns
assert res_replace.loc[0,'valor'] == 40
assert con.execute(f"SELECT MIN(valor) FROM read_parquet('{caminho_parquet.as_posix()}')").fetchone()[0] == 20
print('Projeções validadas; Parquet de origem preservado.')


**Interpretação:** `REPLACE` mudou o resultado da consulta, não o arquivo original; sua coluna continua se chamando `valor`.

## 04.7 — Desafio profissional

**Missão:** identificar o setor com a maior soma de `valor`, calcular a média por evento e verificar se há empates.

**Antes de ver a solução:** escreva sua consulta usando `read_parquet`, `GROUP BY ALL` e ordenação explícita; especifique o resultado esperado.

In [ ]:
# Solução de referência — execute após tentar
resultado_setores = con.execute(f"""
SELECT setor,
       COUNT(*) AS eventos,
       SUM(valor) AS soma_valor,
       ROUND(AVG(valor), 2) AS media_valor
FROM read_parquet('{caminho_parquet.as_posix()}')
GROUP BY ALL
ORDER BY soma_valor DESC, setor
""").df()
display(resultado_setores)
assert resultado_setores['eventos'].sum() == 60
assert resultado_setores['soma_valor'].sum() == con.execute('SELECT SUM(valor) FROM eventos_origem').fetchone()[0]


**Leitura executiva:** a ordenação determina a apresentação; o segundo critério (`setor`) torna o resultado estável em caso de empate. Compare a soma entre origem e Parquet para validar completude.

## 04.8 — Autoavaliação

1. Qual a diferença entre `read_parquet` e `CREATE TABLE AS SELECT`?
2. O que `GROUP BY ALL` faz diante de uma agregação?
3. `SELECT * REPLACE` modifica o arquivo de origem?
4. Por que leitura direta não é necessariamente *zero-copy*?
5. Como provar que o arquivo contém todas as linhas esperadas?

<details><summary>Gabarito comentado</summary>

1. O primeiro consulta o arquivo diretamente; o segundo cria uma tabela persistente ou temporária, conforme o banco e a definição.
2. Agrupa pelas expressões não agregadas da seleção.
3. Não: altera somente a projeção da consulta.
4. Porque podem existir I/O, descompressão, cópias e transformações no processamento.
5. Comparando contagens e totais da origem com os da leitura direta, incluindo verificações de nulos quando aplicáveis.

</details>

## 04.9 — Referências, reprodução e limpeza

- [DuckDB — Parquet Overview](https://duckdb.org/docs/stable/data/parquet/overview)
- [DuckDB — GROUP BY ALL](https://duckdb.org/docs/stable/sql/query_syntax/groupby)
- [DuckDB — SELECT](https://duckdb.org/docs/stable/sql/query_syntax/select)
- [DuckDB — Friendly SQL](https://duckdb.org/docs/stable/sql/dialect/friendly_sql)

**Reprodução:** kernel limpo, execute as células em ordem, confira os `assert` e salve as saídas úteis. O caminho real dos arquivos não é impresso.

**Limpeza:** o arquivo foi criado em pasta temporária e não deve ser publicado no GitHub.

**Estado:** piloto preparado e validado estruturalmente; execução integral no DuckDB **ainda pendente de validação no kernel do projeto**.